# Bias and Fairness Analysis

Comprehensive evaluation of model fairness across demographic groups.

## Analysis Sections
1. Gender Bias Analysis
2. Age Group Bias Analysis
3. Regional Bias Analysis
4. Disparate Impact Analysis

In [ ]:
"""Comprehensive Model Bias AnalysisExamines fairness across demographic groups (gender, age, region)"""import pandas as pdimport numpy as npfrom sklearn.metrics import confusion_matrix, roc_auc_scoreimport json# Load datatest_df = pd.read_csv('data/test.csv')predictions = pd.read_csv('results/mlp_predictions.csv')# Merge predictions with test datadf = test_df.copy()df['pred_calibrated'] = predictions['predicted_probability_calibrated']df['true_label'] = predictions['true_label']print("=" * 80)print("MODEL BIAS ANALYSIS")print("=" * 80)# Gender Analysisprint("\n1. GENDER BIAS ANALYSIS")print("-" * 80)gender_cols = [col for col in df.columns if col.startswith('gender_')]df['gender'] = df[gender_cols].idxmax(axis=1).str.replace('gender_', '')gender_metrics = []for gender in ['Male', 'Female', 'Joint', 'Sex Not Available']:    subset = df[df['gender'] == gender]    if len(subset) > 10:        actual = subset['true_label'].mean()        pred_cal = subset['pred_calibrated'].mean()        try:            auc = roc_auc_score(subset['true_label'], subset['pred_calibrated'])        except:            auc = None        gender_metrics.append({            'Group': gender,            'Sample Size': len(subset),            'Actual Default Rate': actual,            'Predicted (Calibrated)': pred_cal,            'Calibration Gap': abs(actual - pred_cal),            'AUC': auc        })gender_df = pd.DataFrame(gender_metrics)print(gender_df.to_string(index=False))# Age Analysisprint("\n\n2. AGE GROUP BIAS ANALYSIS")print("-" * 80)age_cols = [col for col in df.columns if col.startswith('age_')]df['age_group'] = df[age_cols].idxmax(axis=1).str.replace('age_', '')age_order = ['<25', '25-34', '35-44', '45-54', '55-64', '65-74', '>74']age_metrics = []for age in age_order:    subset = df[df['age_group'] == age]    if len(subset) > 10:        actual = subset['true_label'].mean()        pred_cal = subset['pred_calibrated'].mean()        try:            auc = roc_auc_score(subset['true_label'], subset['pred_calibrated'])        except:            auc = None        age_metrics.append({            'Age Group': age,            'Sample Size': len(subset),            'Actual Default Rate': actual,            'Predicted (Calibrated)': pred_cal,            'Calibration Gap': abs(actual - pred_cal),            'AUC': auc        })age_df = pd.DataFrame(age_metrics)print(age_df.to_string(index=False))# Region Analysisprint("\n\n3. REGIONAL BIAS ANALYSIS")print("-" * 80)region_cols = [col for col in df.columns if col.startswith('region_')]df['region'] = df[region_cols].idxmax(axis=1).str.replace('region_', '')region_metrics = []for region in ['North', 'North-East', 'central', 'south']:    subset = df[df['region'] == region]    if len(subset) > 10:        actual = subset['true_label'].mean()        pred_cal = subset['pred_calibrated'].mean()        try:            auc = roc_auc_score(subset['true_label'], subset['pred_calibrated'])        except:            auc = None        region_metrics.append({            'Region': region,            'Sample Size': len(subset),            'Actual Default Rate': actual,            'Predicted (Calibrated)': pred_cal,            'Calibration Gap': abs(actual - pred_cal),            'AUC': auc        })region_df = pd.DataFrame(region_metrics)print(region_df.to_string(index=False))# Disparate Impactprint("\n\n4. DISPARATE IMPACT ANALYSIS")print("-" * 80)male_subset = df[df['gender'] == 'Male']female_subset = df[df['gender'] == 'Female']male_approval_rate = 1 - (male_subset['pred_calibrated'] >= 0.5).mean()female_approval_rate = 1 - (female_subset['pred_calibrated'] >= 0.5).mean()if male_approval_rate > 0:    gender_di = female_approval_rate / male_approval_rate    print(f"Gender Disparate Impact Ratio: {gender_di:.3f}")    print(f"Passes 80% rule: {'Yes' if gender_di >= 0.8 else 'No'}")# Save resultsbias_results = {    'gender_metrics': gender_df.to_dict('records'),    'age_metrics': age_df.to_dict('records'),    'region_metrics': region_df.to_dict('records')}with open('results/bias_analysis.json', 'w') as f:    json.dump(bias_results, f, indent=4)gender_df.to_csv('results/bias_gender.csv', index=False)age_df.to_csv('results/bias_age.csv', index=False)region_df.to_csv('results/bias_region.csv', index=False)print("\n\nResults saved:")print("  results/bias_analysis.json")print("  results/bias_gender.csv")print("  results/bias_age.csv")print("  results/bias_region.csv")print("\n" + "=" * 80)